In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check CUDA availability and GPU info
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA A40
GPU Memory: 47.70 GB


# Replication: Language Models use Lookbacks to Track Beliefs

This notebook replicates the experiments from "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025).

## Goal
Replicate the key experiments from the belief tracking paper, specifically:
1. Answer Lookback Pointer localization (Fig 4)
2. Answer Lookback Payload localization (Fig 4)
3. Binding Address and Payload localization (Fig 5)
4. Binding Source localization (Fig 6)

## Methodology
- Use interchange interventions with nnsight
- Compare clean and counterfactual prompts
- Measure Interchange Intervention Accuracy (IIA) across layers

## Note on Model Selection
Per the replication guidelines, we will use the smallest available model mentioned in the repository.
The original experiments used Meta-Llama-3-70B-Instruct. Given GPU constraints, we'll attempt to use a smaller model first.

In [3]:
# Load environment variables from bashrc for API keys
import subprocess
result = subprocess.run(['bash', '-c', 'source ~/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        if key in ['HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'NDIF_KEY', 'OPENAI_API_KEY']:
            import os
            os.environ[key] = value
            print(f"Set {key}")

# Verify HF token is set
import os
print(f"HF_TOKEN set: {'HF_TOKEN' in os.environ or 'HUGGINGFACE_TOKEN' in os.environ}")

Set OPENAI_API_KEY
Set HF_TOKEN
HF_TOKEN set: True


In [4]:
# Install required packages
import subprocess
subprocess.run(['pip', 'install', 'nnsight', 'dataclasses_json', '-q'], check=True)
print("Packages installed successfully")

Packages installed successfully


In [5]:
# Core imports
import json
import os
import random
import sys
from dataclasses import dataclass
from collections import defaultdict
from typing import Literal

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from dataclasses_json import DataClassJsonMixin

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Repository paths
REPO_ROOT = "/net/scratch2/smallyan/belief_tracking_eval"
DATA_DIR = os.path.join(REPO_ROOT, "data")
print(f"Repository root: {REPO_ROOT}")

Using device: cuda
Repository root: /net/scratch2/smallyan/belief_tracking_eval


## Data Loading and Dataset Classes

We reimplement the dataset classes from scratch based on the plan and code walkthrough.

In [6]:
# Load story templates
STORY_TEMPLATE_PATH = os.path.join(DATA_DIR, "story_templates.json")
with open(STORY_TEMPLATE_PATH, "r") as f:
    STORY_TEMPLATES = json.load(f)

print("Story templates loaded successfully")
print(f"Number of templates: {len(STORY_TEMPLATES['templates'])}")
print(f"Template 2 (used for experiments):\n{STORY_TEMPLATES['templates'][2]['context']}")

Story templates loaded successfully
Number of templates: 4
Template 2 (used for experiments):
<character_1> and <character_2> are working in a busy restaurant. To complete an order, <character_1> grabs an opaque <container_1> and fills it with <state_1>. Then <character_2> grabs another opaque <container_2> and fills it with <state_2>.


In [7]:
# Load synthetic entities (characters, objects, states)
all_characters = json.load(open(os.path.join(DATA_DIR, "synthetic_entities", "characters.json"), "r"))
all_objects = json.load(open(os.path.join(DATA_DIR, "synthetic_entities", "bottles.json"), "r"))
all_states = json.load(open(os.path.join(DATA_DIR, "synthetic_entities", "drinks.json"), "r"))

print(f"#characters: {len(all_characters)}")
print(f"#objects: {len(all_objects)}")
print(f"#states: {len(all_states)}")
print(f"\nExample characters: {all_characters[:5]}")
print(f"Example objects: {all_objects[:5]}")
print(f"Example states: {all_states[:5]}")

#characters: 103
#objects: 21
#states: 23

Example characters: ['Dean', 'Beth', 'Jake', 'Josh', 'Karen']
Example objects: ['jar', 'cup', 'mug', 'glass', 'flute']
Example states: ['water', 'milk', 'tea', 'beer', 'soda']


In [8]:
# Reimplement Sample dataclass
@dataclass(frozen=False)
class Sample(DataClassJsonMixin):
    """
    Represents a single sample in the CausalToM dataset.
    Each sample contains a story with two characters, two objects, and two states.
    """
    template_idx: int
    characters: list
    objects: list
    states: list
    story: str = None
    character_belief: list = None
    
    def __post_init__(self):
        """Initialize story and beliefs after dataclass creation."""
        if len(self.characters) == 1:
            self.characters.append("<N/A>")
        
        # Ensure no duplicates
        assert len(set(self.states)) == len(self.states), "States must be unique"
        assert len(set(self.objects)) == len(self.objects), "Objects must be unique"
        assert len(set(self.characters)) == len(self.characters), "Characters must be unique"
        
        self._set_story()
    
    def _set_entity_names(self):
        """Replace placeholders in story with actual entity names."""
        # Replace character placeholders
        for i, character in enumerate(self.characters):
            self.story = self.story.replace(
                STORY_TEMPLATES["placeholders"]["entity"]["character"][i], 
                character
            )
        
        # Replace container placeholders
        self.story = self.story.replace(
            STORY_TEMPLATES["placeholders"]["entity"]["container"][0],
            self.objects[0]
        )
        self.story = self.story.replace(
            STORY_TEMPLATES["placeholders"]["entity"]["container"][1],
            self.objects[1]
        )
        
        # Replace state placeholders
        self.story = self.story.replace(
            STORY_TEMPLATES["placeholders"]["entity"]["state"][0], 
            self.states[0]
        )
        self.story = self.story.replace(
            STORY_TEMPLATES["placeholders"]["entity"]["state"][1], 
            self.states[1]
        )
    
    def _set_story(self):
        """Set up the story text and character beliefs."""
        self.template = STORY_TEMPLATES["templates"][self.template_idx]
        self.story = self.template["context"]
        
        # Define world state (ground truth)
        self.world_state = {
            self.objects[0]: self.states[0],
            self.objects[1]: self.states[1],
        }
        
        # Initialize beliefs for both characters
        self.character_belief = [self.world_state.copy(), self.world_state.copy()]
        
        # Set character beliefs based on template
        # Template 2: Characters cannot observe each other's actions
        if self.template_idx in [0, 2, 3]:
            self.character_belief[0][self.objects[1]] = "unknown"
            self.character_belief[1][self.objects[0]] = "unknown"
        elif self.template_idx == 1:
            self.character_belief[1][self.objects[0]] = "unknown"
        
        # Replace placeholders with entity names
        self._set_entity_names()
        
        assert "<" not in self.story and ">" not in self.story, "Placeholders not fully replaced"
        
        return self.story
    
    def __str__(self):
        if self.story is None:
            self._set_story()
        return self.story

# Test Sample class
test_sample = Sample(
    template_idx=2,
    characters=["Alice", "Bob"],
    objects=["bottle", "jar"],
    states=["water", "milk"]
)
print("Test Sample Story:")
print(test_sample.story)
print(f"\nCharacter Beliefs:")
print(f"Alice: {test_sample.character_belief[0]}")
print(f"Bob: {test_sample.character_belief[1]}")

Test Sample Story:
Alice and Bob are working in a busy restaurant. To complete an order, Alice grabs an opaque bottle and fills it with water. Then Bob grabs another opaque jar and fills it with milk.

Character Beliefs:
Alice: {'bottle': 'water', 'jar': 'unknown'}
Bob: {'bottle': 'unknown', 'jar': 'milk'}


In [9]:
# Reimplement Dataset class
@dataclass(frozen=False)
class BeliefDataset(DataClassJsonMixin):
    """
    Dataset class for CausalToM experiments.
    Contains samples and provides methods to generate prompts and answers.
    """
    samples: list
    instruction: str = (
        "1. Track the belief of each character as described in the story. "
        "2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. "
        "3. A character does not have any beliefs about the container and its contents which they cannot observe. "
        "4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. "
        "5. If the queried character has no belief about the container in question, then predict 'unknown'. "
        "6. Do not predict container or character as the final output."
    )
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx, set_container=None, set_character=None):
        """
        Get a formatted prompt and answer for a sample.
        
        Args:
            idx: Sample index
            set_container: Which container to ask about (0 or 1)
            set_character: Which character's belief to query (0 or 1)
        """
        sample = self.samples[idx]
        
        # Randomly choose character and container if not specified
        set_character = random.choice([0, 1]) if set_character is None else set_character
        set_container = random.choice([0, 1]) if set_container is None else set_container
        
        q_actor = sample.characters[set_character]
        belief_states = sample.character_belief[set_character]
        q_container = sample.objects[set_container]
        
        # Get the answer based on the character's belief
        ans = belief_states.get(q_container, "unknown")
        
        # Format the question
        question = sample.template["question"]
        question = question.replace(
            STORY_TEMPLATES["placeholders"]["question"]["character"], 
            q_actor
        )
        question = question.replace(
            STORY_TEMPLATES["placeholders"]["question"]["container"], 
            q_container
        )
        
        # Construct the full prompt
        prompt = f"Instruction: {self.instruction.strip()}\n\n"
        prompt += f"Story: {sample.story.strip()}\n"
        prompt += f"Question: {question}\n"
        prompt += "Answer:"
        
        return {
            "characters": sample.characters,
            "objects": sample.objects,
            "states": sample.states,
            "story": sample.story,
            "question": question,
            "target": ans,
            "prompt": prompt,
            "character_idx": set_character,
            "object_idx": set_container,
            "template_idx": sample.template_idx,
        }

# Test Dataset class
test_dataset = BeliefDataset([test_sample])
result = test_dataset.__getitem__(0, set_container=0, set_character=0)
print("Test Prompt:")
print(result["prompt"])
print(f"\nExpected Answer: {result['target']}")

Test Prompt:
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict container or character as the final output.

Story: Alice and Bob are working in a busy restaurant. To complete an order, Alice grabs an opaque bottle and fills it with water. Then Bob grabs another opaque jar and fills it with milk.
Question: What does Alice believe the bottle contains?
Answer:

Expected Answer: water


## Counterfactual Dataset Generation Functions

We implement the key functions for generating counterfactual samples for the experiments.

In [10]:
def get_reversed_sent_diff_state_counterfacts(
    all_characters: list, 
    all_objects: list, 
    all_states: list, 
    n_samples: int
) -> list:
    """
    Generates counterfactual samples by reversing the sentence order and changing states.
    Used for Answer Lookback Pointer experiment (Fig 4).
    
    Creates pairs where:
    - Clean: Original order with original states
    - Counterfactual: Reversed character/object/state order with different states
    - Target: The state from the other position in clean (i.e., pointer should redirect)
    """
    clean_configs, counterfactual_configs = [], []
    samples = []
    
    for idx in range(n_samples):
        template_idx = 2  # Use template 2 (no visibility)
        characters = random.sample(all_characters, 2)
        objects = random.sample(all_objects, 2)
        states = random.sample(all_states, 2)
        
        sample = Sample(
            template_idx=template_idx,
            characters=characters,
            objects=objects,
            states=states,
        )
        clean_configs.append(sample)
        
        # Create counterfactual with reversed order and different states
        new_states = random.sample(all_states, 2)
        while new_states[0] in states or new_states[1] in states:
            new_states = random.sample(all_states, 2)
        
        sample = Sample(
            template_idx=template_idx,
            characters=list(reversed(characters)),
            objects=list(reversed(objects)),
            states=new_states,
        )
        counterfactual_configs.append(sample)
    
    clean_dataset = BeliefDataset(clean_configs)
    counterfactual_dataset = BeliefDataset(counterfactual_configs)
    
    for idx in range(n_samples):
        random_choice = random.choice([0, 1])
        
        clean = clean_dataset.__getitem__(
            idx,
            set_container=random_choice,
            set_character=random_choice,
        )
        counterfactual = counterfactual_dataset.__getitem__(
            idx,
            set_container=1 ^ random_choice,
            set_character=1 ^ random_choice,
        )
        
        samples.append({
            "clean_characters": clean["characters"],
            "clean_objects": clean["objects"],
            "clean_states": clean["states"],
            "clean_story": clean["story"],
            "clean_question": clean["question"],
            "clean_prompt": clean["prompt"],
            "clean_ans": clean["target"],
            "counterfactual_characters": counterfactual["characters"],
            "counterfactual_objects": counterfactual["objects"],
            "counterfactual_states": counterfactual["states"],
            "counterfactual_story": counterfactual["story"],
            "counterfactual_question": counterfactual["question"],
            "counterfactual_prompt": counterfactual["prompt"],
            "counterfactual_ans": counterfactual["target"],
            "target": " " + clean_configs[idx].states[1 ^ random_choice],
        })
    
    return samples

# Test the function
test_counterfacts = get_reversed_sent_diff_state_counterfacts(all_characters, all_objects, all_states, 2)
print("Example counterfactual pair:")
print(f"Clean answer: {test_counterfacts[0]['clean_ans']}")
print(f"Counterfactual answer: {test_counterfacts[0]['counterfactual_ans']}")
print(f"Target (for IIA): {test_counterfacts[0]['target']}")

Example counterfactual pair:
Clean answer: wine
Counterfactual answer: ale
Target (for IIA):  soda


In [11]:
def get_answer_lookback_payload(
    all_characters: list,
    all_objects: list,
    all_states: list,
    n_samples: int,
) -> list:
    """
    Generates samples for answer lookback payload experiment.
    Creates completely independent clean and counterfactual configurations
    to test if the payload (state token value) can be transferred.
    """
    clean_configs, counterfactual_configs = [], []
    samples = []
    
    for idx in range(n_samples):
        template_idx = 2
        
        # Clean sample
        characters = random.sample(all_characters, 2)
        containers = random.sample(all_objects, 2)
        states = random.sample(all_states, 2)
        sample = Sample(
            template_idx=template_idx,
            characters=characters,
            objects=containers,
            states=states,
        )
        clean_configs.append(sample)
        
        # Counterfactual sample (completely independent)
        characters = random.sample(all_characters, 2)
        containers = random.sample(all_objects, 2)
        states = random.sample(all_states, 2)
        sample = Sample(
            template_idx=template_idx,
            characters=characters,
            objects=containers,
            states=states,
        )
        counterfactual_configs.append(sample)
    
    clean_dataset = BeliefDataset(clean_configs)
    counterfactual_dataset = BeliefDataset(counterfactual_configs)
    
    for idx in range(n_samples):
        random_choice = random.choice([0, 1])
        
        # Clean: ask about the container the character doesn't know about
        clean = clean_dataset.__getitem__(
            idx,
            set_character=random_choice,
            set_container=1 ^ random_choice,  # Other container (unknown)
        )
        
        # Counterfactual: ask about the container the character knows about
        counterfactual = counterfactual_dataset.__getitem__(
            idx,
            set_character=random_choice,
            set_container=random_choice,  # Same position (known)
        )
        
        samples.append({
            "clean_characters": clean["characters"],
            "clean_objects": clean["objects"],
            "clean_states": clean["states"],
            "clean_story": clean["story"],
            "clean_question": clean["question"],
            "clean_ans": clean["target"],
            "clean_prompt": clean["prompt"],
            "counterfactual_characters": counterfactual["characters"],
            "counterfactual_objects": counterfactual["objects"],
            "counterfactual_states": counterfactual["states"],
            "counterfactual_story": counterfactual["story"],
            "counterfactual_question": counterfactual["question"],
            "counterfactual_ans": counterfactual["target"],
            "counterfactual_prompt": counterfactual["prompt"],
            "target": counterfactual["target"],
        })
    
    return samples


def get_reversed_sentence_counterfacts(
    all_characters: list, 
    all_objects: list, 
    all_states: list, 
    n_samples: int
) -> list:
    """
    Generates counterfactual samples by reversing sentences only.
    Used for Binding Address and Payload experiment (Fig 5).
    
    Creates pairs where the same entities are used but in reversed order.
    Target is the state from the other position in clean.
    """
    clean_configs, counterfactual_configs = [], []
    samples = []
    
    for idx in range(n_samples):
        template_idx = 2
        characters = random.sample(all_characters, 2)
        objects = random.sample(all_objects, 2)
        states = random.sample(all_states, 2)
        
        sample = Sample(
            template_idx=template_idx,
            characters=characters,
            objects=objects,
            states=states,
        )
        clean_configs.append(sample)
        
        # Counterfactual: reverse order but keep same entities
        sample = Sample(
            template_idx=template_idx,
            characters=list(reversed(characters)),
            objects=list(reversed(objects)),
            states=list(reversed(states)),
        )
        counterfactual_configs.append(sample)
    
    clean_dataset = BeliefDataset(clean_configs)
    counterfactual_dataset = BeliefDataset(counterfactual_configs)
    
    for idx in range(n_samples):
        random_object_idx = random.choice([0, 1])
        
        clean = clean_dataset.__getitem__(
            idx,
            set_container=random_object_idx,
            set_character=random_object_idx,
        )
        counterfactual = counterfactual_dataset.__getitem__(
            idx,
            set_container=1 ^ random_object_idx,
            set_character=1 ^ random_object_idx,
        )
        
        samples.append({
            "clean_characters": clean["characters"],
            "clean_objects": clean["objects"],
            "clean_states": clean["states"],
            "clean_story": clean["story"],
            "clean_question": clean["question"],
            "clean_prompt": clean["prompt"],
            "clean_ans": clean["target"],
            "counterfactual_characters": counterfactual["characters"],
            "counterfactual_objects": counterfactual["objects"],
            "counterfactual_states": counterfactual["states"],
            "counterfactual_story": counterfactual["story"],
            "counterfactual_question": counterfactual["question"],
            "counterfactual_prompt": counterfactual["prompt"],
            "counterfactual_ans": counterfactual["target"],
            "target": " " + clean_configs[idx].states[1 ^ random_object_idx],
        })
    
    return samples

print("Counterfactual generation functions defined successfully")

Counterfactual generation functions defined successfully


## Model Loading

We will use the smallest available model that can run the experiments. Given the 47GB GPU memory, we'll try to load a smaller version of Llama first.

In [12]:
# Try to load the model
from nnsight import LanguageModel

# Check available shared models
import os
shared_models_dir = "/net/projects/chai-lab/shared_models"
if os.path.exists(shared_models_dir):
    print("Available shared models:")
    for item in os.listdir(shared_models_dir):
        print(f"  - {item}")
else:
    print("Shared models directory not found")

Available shared models:
  - xet
  - hub
  - modules
  - .locks
  - models--meta-llama--Llama-3.3-70B-Instruct
  - models--google--gemma-2-27b-it
  - json
  - token
  - Llama-3.3-70B-Instruct
  - Qwen
  - stored_tokens
  - datasets
  - models--gpt2
  - Meta-Llama-3-8B-Instruct
  - gpt-oss-20b
  - Meta-Llama-3.1-70B-Instruct
  - gemma-2-27b-it


In [13]:
# Per the replication guidelines, use the smallest available model
# The paper uses Llama-3-70B-Instruct, but we have Llama-3-8B-Instruct available which is smaller
# Let's try that first

model_path = "/net/projects/chai-lab/shared_models/Meta-Llama-3-8B-Instruct"
print(f"Loading model from: {model_path}")

# Load with nnsight
model = LanguageModel(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    dispatch=True,
)

print(f"Model loaded successfully!")
print(f"Model config: {model.config.num_hidden_layers} layers, {model.config.hidden_size} hidden dim")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model from: /net/projects/chai-lab/shared_models/Meta-Llama-3-8B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!
Model config: 32 layers, 4096 hidden dim


## Error Detection Function

This function checks if the model correctly answers both clean and counterfactual prompts before using them for IIA measurement.

In [14]:
def error_detection(model, dataloader, is_remote=False):
    """
    Evaluates model performance and identifies errors by comparing 
    predictions on both clean and counterfactual prompts.
    
    Returns:
        tuple: (accuracy, list of error indices)
    """
    correct, total = 0, 0
    errors = []
    
    for bi, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
        clean_prompt = batch["clean_prompt"][0]
        counterfactual_prompt = batch["counterfactual_prompt"][0]
        clean_target = batch["clean_ans"][0]
        counterfactual_target = batch["counterfactual_ans"][0]
        
        with torch.no_grad():
            with model.trace(remote=is_remote) as tracer:
                with tracer.invoke(clean_prompt):
                    clean_pred = model.lm_head.output[0, -1].argmax(dim=-1).item().save()
                
                with tracer.invoke(counterfactual_prompt):
                    counterfactual_pred = model.lm_head.output[0, -1].argmax(dim=-1).item().save()
        
        clean_decoded = model.tokenizer.decode([clean_pred]).lower().strip()
        counterfactual_decoded = model.tokenizer.decode([counterfactual_pred]).lower().strip()
        
        if clean_decoded == clean_target.lower().strip() and \
           counterfactual_decoded == counterfactual_target.lower().strip():
            correct += 1
        else:
            errors.append(bi)
        total += 1
        
        del clean_pred, counterfactual_pred
        torch.cuda.empty_cache()
    
    return correct / total if total > 0 else 0, errors

print("Error detection function defined")

Error detection function defined


## Experiment 1: Answer Lookback Pointer (Fig 4 in Paper)

This experiment tests where the "pointer" information is encoded. When we patch residual streams at the final token position from a counterfactual (with reversed sentence order and different states), we expect the model to output the state from the *other* position in the clean input - demonstrating that the pointer directs to a different state.

In [15]:
# Generate dataset for Answer Lookback Pointer experiment
random.seed(42)
n_samples = 20
batch_size = 1

dataset_pointer = get_reversed_sent_diff_state_counterfacts(
    all_characters,
    all_objects,
    all_states,
    n_samples,
)
dataloader_pointer = DataLoader(dataset_pointer, batch_size=batch_size, shuffle=False)

# Show an example
idx = 0
print("COUNTERFACTUAL EXAMPLE")
print("=" * 50)
print(dataset_pointer[idx]["counterfactual_prompt"][:500], "...")
print(f"\nAnswer: {dataset_pointer[idx]['counterfactual_ans']}")
print()

print("CLEAN EXAMPLE")
print("=" * 50)
print(dataset_pointer[idx]["clean_prompt"][:500], "...")
print(f"\nAnswer: {dataset_pointer[idx]['clean_ans']}")
print(f"\nTarget (what we expect after intervention): {dataset_pointer[idx]['target']}")

COUNTERFACTUAL EXAMPLE
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about ...

Answer: ale

CLEAN EXAMPLE
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the q

In [16]:
# Run error detection to filter out samples where model doesn't answer correctly
_, errors_pointer = error_detection(model, dataloader_pointer)
print(f"Dataset size to be used for IIA: {len(dataset_pointer) - len(errors_pointer)} ({len(errors_pointer)} errors)")

  0%|          | 0/20 [00:00<?, ?it/s]

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  5%|▌         | 1/20 [00:05<01:36,  5.10s/it]

 10%|█         | 2/20 [00:06<00:48,  2.70s/it]

 15%|█▌        | 3/20 [00:07<00:33,  1.94s/it]

...

 90%|█████████ | 18/20 [00:22<00:02,  1.05s/it]

 95%|█████████▌| 19/20 [00:23<00:01,  1.03s/it]

100%|██████████| 20/20 [00:24<00:00,  1.03s/it]

100%|██████████| 20/20 [00:24<00:00,  1.23s/it]

Dataset size to be used for IIA: 2 (18 errors)


In [17]:
# Generate more samples to have enough for valid IIA measurements
random.seed(42)
n_samples = 100  # Increase sample size
batch_size = 1

dataset_pointer = get_reversed_sent_diff_state_counterfacts(
    all_characters,
    all_objects,
    all_states,
    n_samples,
)
dataloader_pointer = DataLoader(dataset_pointer, batch_size=batch_size, shuffle=False)

print(f"Generated {len(dataset_pointer)} samples")

# Run error detection
_, errors_pointer = error_detection(model, dataloader_pointer)
valid_samples = len(dataset_pointer) - len(errors_pointer)
print(f"\nDataset size to be used for IIA: {valid_samples} ({len(errors_pointer)} errors)")
print(f"Accuracy on base task: {(valid_samples/len(dataset_pointer))*100:.1f}%")

Generated 100 samples


  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:01<01:41,  1.03s/it]

  2%|▏         | 2/100 [00:02<01:41,  1.04s/it]

  3%|▎         | 3/100 [00:03<01:39,  1.03s/it]

...

 98%|█████████▊| 98/100 [01:41<00:02,  1.04s/it]

 99%|█████████▉| 99/100 [01:42<00:01,  1.04s/it]

100%|██████████| 100/100 [01:43<00:00,  1.03s/it]

100%|██████████| 100/100 [01:43<00:00,  1.04s/it]


Dataset size to be used for IIA: 3 (97 errors)
Accuracy on base task: 3.0%


In [18]:
# Clear memory and try loading Llama-3.1-70B
del model
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"GPU Memory after clearing: {torch.cuda.memory_allocated()/1e9:.2f} GB used")
print(f"GPU Memory available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

GPU Memory after clearing: 0.01 GB used
GPU Memory available: 47.69 GB


In [19]:
# Try loading the 70B model with quantization to fit in GPU memory
# The 70B model in fp16 needs ~140GB, so we need quantization

try:
    model_path = "/net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct"
    print(f"Attempting to load model from: {model_path}")
    
    # Load with 8-bit quantization to fit in memory
    from transformers import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )
    
    model = LanguageModel(
        model_path,
        device_map="auto",
        quantization_config=quantization_config,
        dispatch=True,
    )
    
    print(f"Model loaded successfully!")
    print(f"Model config: {model.config.num_hidden_layers} layers")
    
except Exception as e:
    print(f"Failed to load 70B model: {e}")
    print("\nFalling back to 8B model...")

Attempting to load model from: /net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded successfully!
Model config: 80 layers


In [20]:
# Now re-run error detection with the 70B model
random.seed(42)
n_samples = 30  # Use fewer samples initially for speed
batch_size = 1

dataset_pointer = get_reversed_sent_diff_state_counterfacts(
    all_characters,
    all_objects,
    all_states,
    n_samples,
)
dataloader_pointer = DataLoader(dataset_pointer, batch_size=batch_size, shuffle=False)

# Run error detection
print("Running error detection on pointer experiment dataset...")
_, errors_pointer = error_detection(model, dataloader_pointer)
valid_samples = len(dataset_pointer) - len(errors_pointer)
print(f"\nDataset size to be used for IIA: {valid_samples} ({len(errors_pointer)} errors)")
print(f"Accuracy on base task: {(valid_samples/len(dataset_pointer))*100:.1f}%")

Running error detection on pointer experiment dataset...


  0%|          | 0/30 [00:00<?, ?it/s]

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/torch/_subclasses/fake_tensor.py:2770: UserWarning: Accessing the data pointer of FakeTensor is deprecated and will error in PyTorch 2.5. This is almost definitely a bug in your code and will cause undefined behavior with subsystems like torch.compile. Please wrap calls to tensor.data_ptr() in an opaque custom op; If all else fails, you can guard accesses to tensor.data_ptr() on isinstance(tensor, FakeTensor). (Triggered internally at /pytorch/c10/core/StorageImpl.cpp:34.)
  return func(*args, **kwargs)


W0116 12:06:41.466000 2052672 site-packages/torch/fx/experimental/symbolic_shapes.py:6679] failed during evaluate_expr(Ne(u0, 0), hint=None, size_oblivious=False, forcing_spec=False


E0116 12:06:41.469000 2052672 site-packages/torch/fx/experimental/recording.py:299] failed while running evaluate_expr(*(Ne(u0, 0), None, False, False), **{})


  0%|          | 0/30 [00:00<?, ?it/s]

GuardOnDataDependentSymNode: Could not guard on data-dependent expression Ne(u0, 0) (unhinted: Ne(u0, 0)).  (Size-like symbols: u0)

ATTENTION: guard_size_oblivious would fix the error, evaluating expression to True.
Maybe you need to add guard_size_oblivious to framework code, see doc below for more guidance.

Caused by: (bitsandbytes/autograd/_functions.py:345 in forward)
For more information, run with TORCH_LOGS="dynamic"
For extended logs when we create symbols, also add TORCHDYNAMO_EXTENDED_DEBUG_CREATE_SYMBOL="u0"
If you suspect the guard was triggered from C++, add TORCHDYNAMO_EXTENDED_DEBUG_CPP=1
For more debugging help, see https://docs.google.com/document/d/1HSuTTVvYH1pTew89Rtpeu84Ht3nQEFTYhAX3Ypa_xJs/edit?usp=sharing

For C++ stack trace, run with TORCHDYNAMO_EXTENDED_DEBUG_CPP=1

In [21]:
# Clear and reload 8B model for faithful replication with smaller model
del model
import gc
gc.collect()
torch.cuda.empty_cache()

# Reload 8B model
model_path = "/net/projects/chai-lab/shared_models/Meta-Llama-3-8B-Instruct"
print(f"Reloading 8B model from: {model_path}")

model = LanguageModel(
    model_path,
    device_map="auto",
    dtype=torch.float16,
    dispatch=True,
)

print(f"Model loaded successfully!")
print(f"Model config: {model.config.num_hidden_layers} layers, {model.config.hidden_size} hidden dim")

Reloading 8B model from: /net/projects/chai-lab/shared_models/Meta-Llama-3-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [22]:
# Check model is loaded
print(f"Model config: {model.config.num_hidden_layers} layers")

NameError: name 'model' is not defined

In [2]:
# Check current state
print(f"Model config: {model.config.num_hidden_layers} layers")

NameError: name 'model' is not defined